# ambient-core — Colab run-me (Bronze → Silver → Gold)

Free Google Colab path. **No Databricks.** Mounts Drive (or uses a local clone),
imports `ambient_pipeline`, runs a pandas smoke against `data/raw/Allmanufacturingds-*.csv`
(academic samples supplied **outside** this GitHub repo), writes under `data/processed/`.

Point `PLATFORM_ROOT` at the [Drive demo folder](https://drive.google.com/drive/folders/1YTmpKrb5J2hqdiD3GJI2cGmFA2cRA1Ne)
or a local clone with external samples downloaded into `data/raw/`.

Commercial / UC / Autoloader notebooks are optional — see `notebooks/COMMERCIAL_ONLY.txt`.

In [ ]:
# Config — change PLATFORM_ROOT if your Drive path differs, or set to a local clone root
PLATFORM_ROOT = "/content/drive/MyDrive/06 Business/Ambient-Systems-Platform"
ORG_ID = "demo-org-allmanufacturingds"

%pip install -q pandas PyYAML

In [ ]:
# Skip this cell when running a local clone (not Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not Colab — using PLATFORM_ROOT as local path")

In [ ]:
import sys
from pathlib import Path

root = Path(PLATFORM_ROOT)
if not (root / "ambient_pipeline").is_dir():
    # Fallback: search under MyDrive, then cwd (local clone)
    hits = []
    for base in (Path("/content/drive/MyDrive"), Path.cwd(), Path.cwd().parent):
        if base.is_dir():
            hits.extend(base.rglob("Ambient-Systems-Platform"))
            hits.extend(p for p in [base] if (p / "ambient_pipeline").is_dir())
    if not hits:
        raise FileNotFoundError(
            "ambient-core root not found. Set PLATFORM_ROOT to Drive folder or local clone."
        )
    root = hits[0]

sys.path.insert(0, str(root))
lib = root / "lib"
if lib.is_dir():
    sys.path.insert(0, str(lib))
print("PLATFORM_ROOT =", root)
print("children:", sorted(p.name for p in root.iterdir()))

In [ ]:
from ambient_pipeline.colab_bootstrap import ensure_package_on_path
from ambient_pipeline.colab_smoke import run_colab_smoke, DEMO_TABLES

root = ensure_package_on_path(root)
raw = root / "data" / "raw"
print("Demo tables expected:")
for tid, fn in DEMO_TABLES.items():
    p = raw / fn
    print(f"  {tid:20} {'OK' if p.is_file() else 'MISSING':7} {fn}")

In [ ]:
result = run_colab_smoke(root, org_id=ORG_ID)
print("run_id:", result.run_id)
print("bronze:", result.bronze_tables)
print("silver:", result.silver_tables)
print("gold KPIs:")
for k, v in result.gold_kpis.items():
    print(f"  {k}: {v}")
print("notes:")
for n in result.notes:
    print(" -", n)
print("wrote →", result.output_dir)

In [ ]:
import pandas as pd

gold = root / "data" / "processed" / "gold" / "demo_kpis.csv"
if gold.is_file():
    display(pd.read_csv(gold))
else:
    print("No gold KPIs yet — place Allmanufacturingds-*.csv under data/raw/ and re-run.")

bronze_sample = root / "data" / "processed" / "bronze" / "ar_aging.csv"
if bronze_sample.is_file():
    print("\nBronze AR (provenance columns present):")
    display(pd.read_csv(bronze_sample).head())

## Next

- Place `Allmanufacturingds-*.csv` samples from Drive (or Ambient Systems demo assets) into `data/raw/`.
- Optional Spark / Delta local path: `lib/ambient_pipeline/` + `pip install -e ".[pipeline]"` (needs external CSVs).
- Skip commercial-only notebooks listed in `notebooks/COMMERCIAL_ONLY.txt`.